In [1]:
!pip install fastapi uvicorn scikit-learn pandas joblib

In [3]:
from google.colab import files

uploaded = files.upload()

Saving customer-churn-training.csv to customer-churn-training (1).csv


In [4]:
import pandas as pd

df = pd.read_csv("customer-churn-training.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (12, 7)


,customer_id,tenure_months,support_tickets,monthly_spend_inr,last_login_days,plan_type,churned
0,C001,2,4,499,21,Basic,1
1,C002,18,1,1299,2,Pro,0
2,C003,6,3,799,14,Basic,1
3,C004,30,0,1499,1,Pro,0
4,C005,11,2,999,5,Standard,0


In [5]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["customer_id", "churned"])
y = df["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Data prepared successfully!")
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Features:", list(X.columns))

Data prepared successfully!
Training rows: 9
Testing rows: 3
Features: ['tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days', 'plan_type']


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [8]:
import joblib

joblib.dump(model, "customer_churn_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [9]:
%%writefile app.py

from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
import joblib

app = FastAPI(title="Customer Churn Prediction API")

model = joblib.load("customer_churn_model.pkl")


class CustomerData(BaseModel):
    tenure_months: int
    support_tickets: int
    monthly_spend_inr: float
    last_login_days: int
    plan_type: str


@app.get("/")
def home():
    return {"message": "Customer Churn Prediction API is running!"}


@app.post("/predict")
def predict_churn(customer: CustomerData):
    data = pd.DataFrame([customer.model_dump()])
    prediction = int(model.predict(data)[0])

    return {
        "churn_prediction": prediction,
        "result": "Likely to Churn" if prediction == 1 else "Not Likely to Churn"
    }

Writing app.py


In [ ]:
!uvicorn app:app --host 127.0.0.1 --port 8000

INFO:     Started server process [6825]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


In [1]:
!pip install fastapi uvicorn scikit-learn pandas joblib

In [2]:
from google.colab import files

uploaded = files.upload()

Saving customer-churn-training.csv to customer-churn-training.csv


In [3]:
import pandas as pd

df = pd.read_csv("customer-churn-training.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (12, 7)


,customer_id,tenure_months,support_tickets,monthly_spend_inr,last_login_days,plan_type,churned
0,C001,2,4,499,21,Basic,1
1,C002,18,1,1299,2,Pro,0
2,C003,6,3,799,14,Basic,1
3,C004,30,0,1499,1,Pro,0
4,C005,11,2,999,5,Standard,0


In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["customer_id", "churned"])
y = df["churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Data prepared successfully!")
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Data prepared successfully!
Training rows: 9
Testing rows: 3


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),

    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols)
])

model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [6]:
import joblib

joblib.dump(model, "customer_churn_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [7]:
%%writefile app.py

from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
import joblib

app = FastAPI(title="Customer Churn Prediction API")

model = joblib.load("customer_churn_model.pkl")


class CustomerData(BaseModel):
    tenure_months: int
    support_tickets: int
    monthly_spend_inr: float
    last_login_days: int
    plan_type: str


@app.get("/")
def home():
    return {"message": "Customer Churn Prediction API is running!"}


@app.post("/predict")
def predict_churn(customer: CustomerData):
    data = pd.DataFrame([customer.model_dump()])
    prediction = int(model.predict(data)[0])

    return {
        "churn_prediction": prediction,
        "result": "Likely to Churn" if prediction == 1 else "Not Likely to Churn"
    }

Writing app.py


In [10]:
!nohup uvicorn app:app --host 127.0.0.1 --port 8000 > server.log 2>&1 &

In [11]:
!cat server.log

INFO:     Started server process [2500]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [12]:
!cat server.log

INFO:     Started server process [2500]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [13]:
import requests

response = requests.get("http://127.0.0.1:8000/", timeout=10)

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'message': 'Customer Churn Prediction API is running!'}


In [14]:
import requests

url = "http://127.0.0.1:8000/predict"

customer = {
    "tenure_months": 12,
    "support_tickets": 3,
    "monthly_spend_inr": 1500.0,
    "last_login_days": 10,
    "plan_type": "Basic"
}

response = requests.post(url, json=customer)

print("Status code:", response.status_code)
print("Prediction:", response.json())

Status code: 200
Prediction: {'churn_prediction': 0, 'result': 'Not Likely to Churn'}


In [15]:
%%writefile requirements.txt
fastapi
uvicorn
scikit-learn
pandas
joblib
pydantic

Writing requirements.txt


In [16]:
from google.colab import files

files.download("app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>